<a href="https://colab.research.google.com/github/sheaqiu/lecture-ai-engineering-2026/blob/lec02/notebooks/lora_glossary_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colabで開く"/></a>

# 第02回 Advanced：用語集を用いたLoRA微調整

このノートブックは講義の元の版をもとに、データ、プロンプト、評価を変更しています。モデルと依存ライブラリの固定バージョンは講義の設定を使います。

学習前評価 → LoRA学習 → 同じ条件での学習後評価 → 用語検索との比較 → 結果の書き出し。

モデルは `Qwen/Qwen3.5-0.8B` で、T4 GPUを使い、量子化なしのLoRAを行います。3エポックは最初の実験として事前に決めた設定で、データ量は元のFAQ実験より多くなっています。

実験は公式英語版の `matsuolab/lecture-ai-engineering-2026-en` をforkして実行しました。資料のコミットは `2a89014410768f2307a72d0727d1c6bdb29cd726` です。提出先は日本語版のforkに移しましたが、学習・評価用の英語データ、共通のシステム指示、採点規則と保存済みの元の結果は変更していません。このノートブックは `lectures/02/advanced/` の実験用スナップショットを読み込みます。日本語版直下の用語集を使う実験ではありません。

[資料の出典と保存方針](../lectures/02/source_english/README.md)／[演習・課題レポート](../lectures/02/answer.md)

## 0. T4 GPUと講義の依存ライブラリ

In [ ]:
assert not any(name in globals() for name in ('model', 'tuned', 'trainer')), 'セッションを再起動し、第0節から実行してください。'
import subprocess
import torch
assert torch.cuda.is_available(), 'ランタイムの種類でT4 GPUを選択してください。'
print('GPU:', torch.cuda.get_device_name(0))
print('Colab torch:', torch.__version__)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
# 講義のPINSを使い、torchはColabに付属するものを利用します。
PINS = {'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0'}
spec = ' '.join(f'"{key}=={value}"' for key, value in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets
print({module.__name__: module.__version__ for module in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 実験条件の固定

新しい出力ディレクトリを使い、元のFAQ実験や他の実行のファイルが混ざらないようにします。

学習前後で同じプロンプト、18問のテスト、貪欲生成、32-bitの基礎モデルを使用します。

In [ ]:
import json, time, math, hashlib, csv, shutil, uuid
from datetime import datetime, timezone
from pathlib import Path
from transformers import set_seed
import pandas as pd
from IPython.display import display

CONFIG = {
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'revision': '2fc06364715b967f1860aea9cf38778875588b17',
    'seed': 42, 'num_train_epochs': 3, 'warmup_steps': 6,
    'lr': 2e-4, 'lora_r': 16, 'lora_alpha': 32,
    'max_seq_len': 512, 'max_new_tokens': 192,
    'gen': {'do_sample': False},
}
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/advanced_glossary') / RUN_ID
MATERIALS = WORK / 'materials'
MATERIALS.mkdir(parents=True, exist_ok=False)
set_seed(CONFIG['seed'])
pd.set_option('display.max_colwidth', None)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print('出力ディレクトリ：', WORK)

## 2. データの読み込み、分割の確認と検索による比較

元のスクリプトは58用語に2通りずつの質問を作り、116件の問答を生成します。各記録に共通のシステム指示を付け、定義をJSONに変換し、未掲載用語の学習例を8件追加します。学習データは合計124件です。

検証用は12件、テスト用は18件です。分割間で質問文は重複しません。既知概念は共有し、新しい聞き方への対応と対象範囲の判断を調べます。未掲載用語は学習・検証・テストで別々に選びます。

「未掲載」は用語集に項目がないという意味です。その概念が存在しない、または基礎モデルが知らないという意味ではありません。本タスクでは、これらの質問に対して確認を求めることを要求します。

モデルへの評価入力は質問と共通のシステム指示だけで、定義や正解は与えません。比較用の検索処理は用語集にアクセスし、用語名・別名に一致する定義を返します。単純な検索では言い換えを見逃す場合もあります。

In [ ]:
import urllib.request
import importlib.util
BASE_URL = 'https://raw.githubusercontent.com/sheaqiu/lecture-ai-engineering-2026/lec02/lectures/02/advanced/'
FILE_HASHES = {
    "README.md": "6defb544aa7e2e0c674c2400c6ca2238c1d8fbd345d7543b1a44536cc4b9987a",
    "advanced_glossary.py": "01ec16f695428e5bec9092d740a7f8c1d2121a1a16cd4b81589d5bed69096a5d",
    "glossary.json": "c3810badb201b5dd44dd9476189297a468567581b1368b82973791a0b709e9d5",
    "glossary_sft.jsonl": "57e84666ee5e263c51300b2ba8565481f437bc71288acebc76f72e413fa34d3f",
    "gold.json": "09a7ec647c834c11a559235c94fa9b30e9d22f5f61bc34ecf1dc5930f440dfd2",
    "manifest.json": "6722f5de4dca308aa95c86d88c8f02522cbdb63ab7790916f812c4409db063c7",
    "test.jsonl": "d754fc24b139f63bab6b46488792b83aa66d00cc96bfb0d7116d84bc51437ca4",
    "train.jsonl": "81c58283cbd84a6e322b22caa9565ccecba74849fa6a273f8926f3110d13b5e6",
    "valid.jsonl": "ba1c209907107db2abc50a141f2172f5ac2eb1e379ead93d0f8dcf82568a0f1f"
}
for name, expected_hash in FILE_HASHES.items():
    with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
        payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == expected_hash, f'ファイルの検証に失敗：{name}'
    (MATERIALS / name).write_bytes(payload)

module_spec = importlib.util.spec_from_file_location('aie_advanced', MATERIALS / 'advanced_glossary.py')
helper = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(helper)
glossary = json.loads((MATERIALS / 'glossary.json').read_text())
raw_records = [json.loads(line) for line in (MATERIALS / 'glossary_sft.jsonl').read_text().splitlines()]
prepared = helper.make_materials(glossary, raw_records)
train, valid, test, gold = [prepared[name] for name in ['train', 'valid', 'test', 'gold']]
for name in ['train', 'valid', 'test']:
    loaded = [json.loads(line) for line in (MATERIALS / f'{name}.jsonl').read_text().splitlines()]
    assert loaded == prepared[name]
assert json.loads((MATERIALS / 'gold.json').read_text()) == gold
print(f'glossary={len(glossary["terms"])} / raw={len(raw_records)}')
print(f'train={len(train)} / valid={len(valid)} / test={len(test)}')
print('学習例：')
print(json.dumps(train[0], ensure_ascii=False, indent=2))
display(pd.DataFrame([{'Category': row['category'], 'Question': row['question'],
                      'Expected needs_review': gold[row['id']]['needs_review'],
                      'Reference answer': gold[row['id']]['reference_answer']} for row in test]))

In [ ]:
retrieval_summary, retrieval_rows = helper.evaluate(
    lambda question: helper.lookup_answer(question, glossary), test, gold, 'retrieval', WORK)
display(pd.DataFrame([retrieval_summary]).drop(columns=['by_category']))
display(pd.DataFrame([{'Question': row['question'], 'Lookup output': row['raw'], 'Content pass': row['content_ok']}
                      for row in retrieval_rows]))
print('第0–2節が完了しました。学習例、18問と参照回答を確認してから第3節へ進んでください。')

## 3. 基礎モデルを読み込み、学習前の性能を評価

モデルのrevisionを固定し、量子化していない32-bitの重みを使います。最初に学習記録のトークン長を確認し、512トークンの上限によって回答が切り落とされないことを確かめます。

厳密なJSON形式の評価と、キーワードによる内容評価を分けて報告します。内容評価では同じ要点の一般的な語形を複数許容しますが、回答内容の確認に代わるものではありません。最後にすべての元の回答を書き出します。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
lengths = [len(tok.apply_chat_template(row['messages'], tokenize=True, return_dict=False)) for row in train + valid]
assert max(lengths) <= CONFIG['max_seq_len'], f'学習記録が長すぎます：{max(lengths)}。データと長さの上限を確認してください。'
TOKEN_LENGTHS = {'max_train_valid_tokens': max(lengths), 'limit': CONFIG['max_seq_len']}
print('Token lengths:', TOKEN_LENGTHS)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to('cuda')
model.config.use_cache = True
model.eval()
print('Model weight memory (GB):', round(model.get_memory_footprint() / 1e9, 2))

def generate(active_model, question):
    prompt = tok.apply_chat_template(helper.messages(question), tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt', add_special_tokens=False).to(active_model.device)
    active_model.eval()
    with torch.no_grad():
        output = active_model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
                                       pad_token_id=tok.pad_token_id, **CONFIG['gen'])
    return tok.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

before_summary, before_rows = helper.evaluate(lambda question: generate(model, question), test, gold, 'before', WORK)
display(pd.DataFrame([{'Question': row['question'], 'Before output': row['raw'],
                      'Format pass': row['format_ok'], 'Content pass': row['content_ok']} for row in before_rows]))
print(json.dumps(before_summary, ensure_ascii=False, indent=2))

## 4. LoRA学習

124件の学習記録を3エポック学習します。バッチ4 × 勾配蓄積2で、1エポック当たり16回、合計48回の更新を見込みます。実際の出力で確認してください。

LoRAを適用する層と学習率は講義の設定を使います。学習時間、学習全体の平均loss、最後のログ区間のloss、学習後の検証lossを記録します。検証データは勾配更新に使いません。

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

set_seed(CONFIG['seed'])
train_ds, valid_ds = Dataset.from_list(train), Dataset.from_list(valid)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'], lora_dropout=0.05,
                  bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
sft = SFTConfig(output_dir=str(WORK / 'ckpt'), num_train_epochs=CONFIG['num_train_epochs'],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG['lr'], lr_scheduler_type='cosine', warmup_steps=CONFIG['warmup_steps'],
                logging_steps=4, save_steps=16, save_total_limit=1,
                fp16=False, bf16=False, max_length=CONFIG['max_seq_len'], packing=False,
                report_to='none', seed=CONFIG['seed'])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds,
                     peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
torch.cuda.synchronize()
training_start = time.perf_counter()
train_output = trainer.train()
torch.cuda.synchronize()
training_seconds = time.perf_counter() - training_start
assert trainer.state.global_step > 0 and math.isfinite(float(train_output.training_loss)), '学習が正常に完了しませんでした。'
loss_logs = [row for row in trainer.state.log_history if 'loss' in row]
validation_metrics = trainer.evaluate()
TRAINING = {'seconds': round(training_seconds, 2), 'steps': trainer.state.global_step,
            'epochs': trainer.state.epoch, 'train_loss_average': float(train_output.training_loss),
            'last_logged_loss': loss_logs[-1]['loss'] if loss_logs else None,
            'last_loss_log_step': loss_logs[-1]['step'] if loss_logs else None,
            'validation_loss': validation_metrics.get('eval_loss'),
            'train_records': len(train), 'valid_records': len(valid)}
tuned = trainer.model
tuned.save_pretrained(WORK / 'adapter')
tok.save_pretrained(WORK / 'adapter')
tuned.config.use_cache = True
tuned.eval()
(WORK / 'training.json').write_text(json.dumps(TRAINING, indent=2), encoding='utf-8')
(WORK / 'training_log.json').write_text(json.dumps(trainer.state.log_history, indent=2), encoding='utf-8')
print(json.dumps(TRAINING, indent=2))

## 5. 同じ条件での学習後評価と誤りの分析

テスト問題、プロンプト、生成パラメータは変更しません。`model` にもLoRAが追加されているため、学習前の比較には保存済みの回答を使います。基礎モデルの回答を再生成する場合は `with tuned.disable_adapter():` を使います。

3種類の問題ごとの結果と、学習後に悪化した問題を確認します。検索処理のJSONはプログラムが構成するため、その形式合格率はモデルの能力ではありません。回答内容、対象範囲の扱い、言い換えへの対応、保守方法を中心に比較します。

In [ ]:
after_summary, after_rows = helper.evaluate(lambda question: generate(tuned, question), test, gold, 'after', WORK)
assert [row['id'] for row in before_rows] == [row['id'] for row in after_rows]
regressed = [before['question'] for before, after in zip(before_rows, after_rows)
             if before['both_ok'] and not after['both_ok']]
display(pd.DataFrame([before_summary, after_summary, retrieval_summary])[
    ['label', 'n', 'format_rate', 'content_rate', 'both_rate', 'seconds']])
display(pd.DataFrame([{'system': summary['label'], 'category': category, **scores}
                      for summary in [before_summary, after_summary, retrieval_summary]
                      for category, scores in summary['by_category'].items()]))
display(pd.DataFrame([{'Question': before['question'], 'Expected review': gold[before['id']]['needs_review'],
                      'Before output': before['raw'], 'After output': after['raw'],
                      'Before content pass': before['content_ok'], 'After content pass': after['content_ok']}
                     for before, after in zip(before_rows, after_rows)]))
print('自動評価で悪化した質問：', regressed)

## 6. 結果の書き出しと内容の確認

出力ZIPにはデータ、プロンプト、設定、3種類の逐問結果、学習ログ、内容確認用の表、レポート案が含まれます。モデルの重みとチェックポイントは実行環境のディレクトリに残り、結果ZIPには含まれません。

ZIPをダウンロードしてCodexでの分析に使い、元の回答は自分でも確認してください。

In [ ]:
source_manifest = json.loads((MATERIALS / 'manifest.json').read_text())
result = {
    'experiment': 'Lecture 02 Advanced: course glossary',
    'config': CONFIG,
    'versions': {module.__name__: module.__version__ for module in [transformers, peft, trl, accelerate, datasets, torch]},
    'method': 'LoRA (no quantization; float32 base weights)',
    'source_manifest': source_manifest, 'materials_sha256': FILE_HASHES,
    'prompt': helper.SYSTEM, 'token_lengths': TOKEN_LENGTHS,
    'before': before_summary, 'after': after_summary, 'retrieval': retrieval_summary,
    'regressed': regressed, 'training': TRAINING,
    'executed_training_myself': True,
    'manual_review_status': 'pending: inspect raw answers and complete manual_review.csv',
}
(WORK / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
review_rows = []
for before, after, lookup in zip(before_rows, after_rows, retrieval_rows):
    review_rows.append({'id': before['id'], 'category': before['category'], 'question': before['question'],
        'reference_answer': gold[before['id']]['reference_answer'], 'expected_needs_review': gold[before['id']]['needs_review'],
        'before_raw': before['raw'], 'after_raw': after['raw'], 'lookup_raw': lookup['raw'],
        'before_keyword_pass': before['content_ok'], 'after_keyword_pass': after['content_ok'],
        'lookup_keyword_pass': lookup['content_ok'], 'manual_verdict': '', 'reason': ''})
with (WORK / 'manual_review.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(review_rows[0]))
    writer.writeheader()
    writer.writerows(review_rows)

report_lines = [
    '# 第2回 Advanced：課題レポートの下書き', '',
    '- 取り組んだレベル：Advanced',
    '- 出典：公式英語版をforkして実験。コミット 2a89014410768f2307a72d0727d1c6bdb29cd726。提出先は日本語版のfork。',
    '- データ：英語の用語集58項目。元の生成スクリプトによる116件をJSON回答へ変換し、未掲載用語の学習例8件を追加。',
    '- 分割：学習124件、検証12件、テスト18件（用語名6件、言い換え6件、未掲載用語6件）。',
    '- モデルは用語集を入力せずに回答。未掲載かどうかは用語集の項目で判定し、モデルの一般知識とは区別。',
    '- 事前に固定した設定：3エポック、rank=16、学習率2e-4、量子化なしのfloat32基礎モデル。',
    '- AIの利用：コード作成、問題案、評価規則の整理にCodexを補助として利用。本人による確認内容：要記入。', '',
    '| 方法 | 形式の合格率 | キーワード内容合格率 | 両方の合格率 |',
    '|---|---:|---:|---:|',
]
for summary in [before_summary, after_summary, retrieval_summary]:
    method_name = {'before': '学習前', 'after': '学習後', 'retrieval': '単純な用語検索'}[summary['label']]
    report_lines.append(f"| {method_name} | {summary['format_rate']:.2%} | {summary['content_rate']:.2%} | {summary['both_rate']:.2%} |")
report_lines += ['', f"学習時間（秒）：{TRAINING['seconds']}",
                 f"学習全体の平均loss：{TRAINING['train_loss_average']}",
                 f"最後のログ区間のloss：{TRAINING['last_logged_loss']}",
                 f"自動評価で悪化した質問：{json.dumps(regressed, ensure_ascii=False)}", '',
                 '## 元の回答を確認してから記入する項目', '',
                 '- 学習後に残った誤りと確認フラグの誤判断：要記入。',
                 '- キーワード採点と回答内容の正しさが一致しなかった例：要記入。',
                 '- 単純な用語検索と比べた長所と短所：要記入。',
                 '- 微調整が向く場面と検索が向く場面：要記入。',
                 '- 限界：小規模な固定テスト、既知概念の共有、キーワード評価、単純な語の一致による検索。',
                 '- 元のFAQ演習は、別途レポートの演習欄に記載する。', '']
(WORK / 'answer_draft.md').write_text('\n'.join(report_lines), encoding='utf-8')

import zipfile
archive = WORK.parent / f'advanced_results_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(WORK.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
    for path in sorted(MATERIALS.iterdir()):
        if path.is_file():
            bundle.write(path, 'materials/' + path.name)
print('結果のZIP：', archive)
print('ZIPのダウンロードと保存を確認したら、Colabのランタイムを切断して削除してください。')
from google.colab import files
files.download(str(archive))